# Fase 2.3 — Spearman definitivo

## 1. Objetivo y fuentes

Se documenta una entrada parsimoniosa para predecir el frío del bloque siguiente. Se leen artefactos existentes sin búsquedas, entrenamiento ni acceso a Test. Las cifras se obtienen de esos archivos.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "consigna.md").is_file())
sys.path.insert(0, str(raiz / "src"))
from auxiliar_functions import tabla_simple
from seleccion_definitiva import cargar_seleccion_definitiva
oficial = cargar_seleccion_definitiva()
base = raiz / "results/seleccion"
individuales = json.loads((base / "ejecucion-20261113T184058Z-969d4545/selecciones_por_metodo.json").read_text(encoding="utf-8"))
forward = json.loads((base / "forward_max60_artefactos/resultado.json").read_text(encoding="utf-8"))
resultados = json.loads((base / "comparacion_validation_congelada/resultados.json").read_text(encoding="utf-8"))
candidatos = json.loads((base / "candidatos_congelados_validation/candidatos.json").read_text(encoding="utf-8"))
from zipfile import ZipFile
import hashlib
from variables_interacciones import construir_mapa_interacciones
portabilidad = json.loads((base / "cierre_fase23/portabilidad.json").read_text(encoding="utf-8"))
with ZipFile(base / "historial/ejecucion-20261113T184058Z-969d4545-evidencia.zip") as archivo:
    resumen_interno = pd.read_csv(archivo.open("ejecucion-20261113T184058Z-969d4545/resumen_resultados.csv"))
progreso_forward = json.loads((base / "forward_max60_artefactos/progreso.json").read_text(encoding="utf-8"))

## 2. Métodos y separación temporal

Train es el prefijo antiguo; Validation compara candidatos congelados y Test se reserva al final. Spearman usa asociación monotónica, Pearson lineal, información mutua dependencia general y Lasso coeficientes activos. Los filtros y Lasso se buscaron con pliegues externos e internos temporales; Forward tradicional agrega candidatas desde vacío con Ridge y límite de columnas efectivas. Se aprenden reglas por prefijo y se protege la etiqueta del bloque siguiente mediante gap. El informe detalla procedimientos, jerarquía e historial descartado.

In [2]:
filas = [{"método": m, "variables": len(v["seleccionadas"]), "configuración": str(v["configuracion"])} for m, v in individuales.items()]
filas.append({"método": "forward", "variables": forward["cantidad_directa"], "configuración": "Forward tradicional; Ridge alpha=100; max60"})
print("Tabla 1. Cinco métodos y configuraciones históricas")
display(tabla_simple(pd.DataFrame(filas)))

Tabla 1. Cinco métodos y configuraciones históricas


,método,variables,configuración
0,spearman,14,"{'metodo': 'spearman', 'escalador': 'standard', 'transformacion': 'ninguna', 'k': 11}"
1,pearson,19,"{'metodo': 'pearson', 'escalador': 'standard', 'transformacion': 'ninguna', 'k': 15}"
2,mutua,311,"{'metodo': 'mutua', 'escalador': 'standard', 'transformacion': 'yeo-johnson', 'k': 311}"
3,lasso,268,"{'metodo': 'lasso', 'escalador': 'standard', 'transformacion': 'yeo-johnson', 'alpha': 3.1622776601683795}"
4,forward,30,Forward tradicional; Ridge alpha=100; max60


## 3. Resultados de Validation

La comparación usa las mismas observaciones y preparación aprendida con Train. Pearson obtuvo el menor MAE absoluto. Se elige Spearman por parsimonia en el margen fijado antes de evaluar. Validation participó de la decisión y no es una estimación final independiente.

In [3]:
tabla = pd.DataFrame(oficial["comparacion"])[["metodo", "variables", "columnas_efectivas", "mae", "filas"]]
tabla["seleccionado"] = tabla.metodo.eq(oficial["metodo"])
print("Tabla 2. Comparación congelada de Validation")
display(tabla_simple(tabla))
minimo = tabla.mae.min()
print(oficial["criterio"])
print(f"Diferencia: {oficial['mae_validation'] - minimo:.6f} kW; {(oficial['mae_validation']/minimo-1)*100:.4f} %")
print("Criterio previo:", candidatos["criterio_previo"])
print("Fecha del congelamiento:", candidatos["fecha"])

Tabla 2. Comparación congelada de Validation


,metodo,variables,columnas_efectivas,mae,filas,seleccionado
0,pearson,19,37,2239.58,176,False
1,spearman,14,27,2247.82,176,True
2,mutua,311,619,2736.27,176,False
3,lasso,268,532,2723.35,176,False
4,forward,30,59,2377.22,176,False


Dentro del 1% del MAE mínimo, preferir menos variables; criterio congelado antes de evaluar.
Diferencia: 8.237117 kW; 0.3678 %
Criterio previo: MAE mínimo, empate práctico dentro del 1%: menor cantidad de variables; mismo Ridge y filas
Fecha del congelamiento: 2026-11-13T22:35:18.737995+00:00


## 4. Conjunto oficial y preprocesamiento

Nombres y orden se leen del contrato definitivo. Las columnas efectivas agregan indicadores de ausencia de Train a las variables; el calendario no tiene indicador. Los rezagos usan bloques anteriores, la media termina hoy y los productos usan el mapa congelado. El coseno usa la etiqueta del día siguiente. Se reutilizan limpieza e ingeniería existentes sin aprender reglas nuevas.

In [4]:
print("Tabla 3. Variables oficiales en orden e indicadores de Train")
display(tabla_simple(pd.DataFrame({"orden": range(1, len(oficial["variables"])+1), "variable": oficial["variables"], "indicador": [c in oficial["preprocesamiento"]["indicadores"] for c in oficial["variables"]]})))
print("Columnas efectivas:", len(oficial["variables"]) + len(oficial["preprocesamiento"]["indicadores"]))

Tabla 3. Variables oficiales en orden e indicadores de Train


,orden,variable,indicador
0,1,Frio (Kw),True
1,2,Frio (Kw) x frio_hace_2_dias,True
2,3,frio_media_7_dias,True
3,4,Sala Maq (Kw),True
4,5,Frio (Kw) x Consolidado Agua | Agua Planta (Hl),True
5,6,Servicios (Kw),True
6,7,Frio (Kw) x Totalizadores Energia | KW Trafo 9,True
7,8,dia_anio_coseno,False
8,9,frio_hace_2_dias x Consolidado Agua | Agua Planta (Hl),True
9,10,dia_anio_coseno x Consolidado Agua | Agua Planta (Hl),True


Columnas efectivas: 27


## 5. Controles de artefactos

Se verifica igualdad de listas, preparación y resultados guardados. Estos controles no certifican horarios físicos. No se ejecutan selección, comparadores ni ajuste de modelos.

In [5]:
mapa = construir_mapa_interacciones(oficial["ingenieria"])
controles = {
    "Jerarquía completa": all(set(mapa.get(c, [])).issubset(oficial["variables"]) for c in oficial["variables"]),
    "Huella histórica de candidatos": resultados["candidatos_sha256"] == portabilidad["candidatos_sha256_original"],
    "Huella de metadatos portables": hashlib.sha256((base / "candidatos_congelados_validation/candidatos.json").read_bytes()).hexdigest() == portabilidad["candidatos_sha256_actual"],
    "Spearman oficial": oficial["metodo"] == "spearman",
    "Lista y orden congelados": oficial["variables"] == candidatos["conjuntos"]["spearman"] == individuales["spearman"]["seleccionadas"],
    "Mismos índices Validation": oficial["indices_validation"] == resultados["indices_validation"],
    "Soporte común": all(r["filas"] == oficial["observaciones"] for r in resultados["resultados"]),
    "Preparación congelada": oficial["preprocesamiento"] == next(r["preparacion"] for r in resultados["resultados"] if r["metodo"] == oficial["metodo"]),
    "Test reservado en contrato": oficial["test_utilizado"] is False,
    "Parsimonia en margen previo": oficial["mae_validation"] <= 1.01 * minimo,
}
assert all(controles.values())
print("Tabla 4. Controles de coherencia de artefactos")
display(tabla_simple(pd.DataFrame({"control": controles.keys(), "cumple": controles.values()})))

Tabla 4. Controles de coherencia de artefactos


,control,cumple
0,Jerarquía completa,True
1,Huella histórica de candidatos,True
2,Huella de metadatos portables,True
3,Spearman oficial,True
4,Lista y orden congelados,True
5,Mismos índices Validation,True
6,Soporte común,True
7,Preparación congelada,True
8,Test reservado en contrato,True
9,Parsimonia en margen previo,True


## 6. Baselines sobre soporte común

Se comparan únicamente las observaciones comunes guardadas para el modelo y las dos referencias. El MAE del modelo en toda Validation no se mezcla con las referencias de menor soporte. La mejora se calcula frente al mejor baseline y sigue siendo una estimación utilizada durante la selección.

In [6]:
spearman = next(r for r in oficial["comparacion"] if r["metodo"] == oficial["metodo"])
base_ref = oficial["baselines"]
soporte = base_ref["persistencia"]["indices"]
assert soporte == base_ref["media_movil"]["indices"]
assert set(soporte).issubset(oficial["indices_validation"])
print("Tabla 5. Spearman y baselines en las mismas filas comunes")
display(tabla_simple(pd.DataFrame([
    {"método": "Spearman", "MAE kW": spearman["mae_en_filas_baseline"], "filas": len(soporte)},
    *[{"método": k, "MAE kW": v["mae"], "filas": v["filas"]} for k, v in base_ref.items()],
])))
mejor_baseline = min(v["mae"] for v in base_ref.values())
print(f"Mejora frente al mejor baseline: {(1-spearman['mae_en_filas_baseline']/mejor_baseline)*100:.4f} %")

Tabla 5. Spearman y baselines en las mismas filas comunes


,método,MAE kW,filas
0,Spearman,2173.40,171
1,persistencia,2668.33,171
2,media_movil,2497.65,171


Mejora frente al mejor baseline: 12.9825 %


## 7. Forward tradicional y costo registrado

El procedimiento parte del conjunto vacío, evalúa candidatas por nombre con Ridge, respeta pliegues temporales con separación y limita la dimensión efectiva incluyendo indicadores. El error mostrado es interno: guio la selección y no estima rendimiento independiente. El último checkpoint documenta una ronda posterior sin mejora. La duración procede del registro, no de una estimación nueva.

In [7]:
checkpoint_posterior = json.loads((base / "forward_max60_artefactos/checkpoint_ronda_31.json").read_text(encoding="utf-8"))
print("Tabla 6. Mejor ronda Forward y tiempo de ejecución")
display(tabla_simple(pd.DataFrame([{
    "ronda elegida": forward["ronda"], "variables": forward["cantidad_directa"],
    "MAE interno kW": forward["mae_interno"], "columnas por pliegue": str(forward["columnas_ridge"]),
    "ronda posterior": checkpoint_posterior["ronda"],
    "MAE ronda posterior": checkpoint_posterior["ganador"]["mae"],
    "duración segundos": progreso_forward["segundos_transcurridos"],
}])))

Tabla 6. Mejor ronda Forward y tiempo de ejecución


,ronda elegida,variables,MAE interno kW,columnas por pliegue,ronda posterior,MAE ronda posterior,duración segundos
0,30,30,2826.52,"[39, 59, 59]",31,2834.76,449.88


## 8. Convergencia de Lasso y alcance del evaluador

Se conserva la advertencia externa histórica y se distingue de la comprobación posterior del ajuste final. No se repite el ajuste. La selección usa un evaluador Ridge con regularización fija; el Pipeline del equipo explora otros valores de alpha. El conjunto podría cambiar para otra regularización o modelo, y eso limita el alcance de la comparación.

In [8]:
registro_lasso = candidatos["lasso"]
print("Tabla 7. Comprobación final de Lasso registrada")
display(tabla_simple(pd.DataFrame([{
    "dual_gap": registro_lasso["dual_gap"], "tolerancia efectiva": registro_lasso["umbral"],
    "iteraciones": registro_lasso["iteraciones"],
    "cumple tolerancia": registro_lasso["dual_gap"] < registro_lasso["umbral"],
    "selección reproducida": registro_lasso["seleccion_reproducida"],
    "salvedad histórica": registro_lasso["advertencia"],
}])))

Tabla 7. Comprobación final de Lasso registrada


,dual_gap,tolerancia efectiva,iteraciones,cumple tolerancia,selección reproducida,salvedad histórica
0,6102.32,6108.62,8630,True,True,pliegue externo previo no convergió


## 9. Origen y regeneración explícita de resultados

La ejecución normal del notebook lee únicamente artefactos incluidos en el manifiesto de Git y el paquete de evidencia. No usa Excel, cachés privados ni rutas fuera del repositorio. Para regenerar búsquedas se necesitan los originales del aula virtual, comprobados con data/checksums.json, y el entorno cervecera_env. Los comandos de la tabla son documentación y no se ejecutan en este notebook.

El motor de los métodos individuales usa Train; sus tiempos por etapa se leen del resumen histórico. Forward dispone de comando independiente y una salida nueva. El comparador usa los candidatos ya congelados y guarda una reproducción en otro directorio. No hay un tiempo medido de esa comparación y no se inventa uno. El JSON oficial es un contrato congelado versionado, no un resultado que el notebook vuelva a seleccionar: se comprueba contra las listas y resultados asociados. Los comandos nuevos usan la interfaz del punto de entrada integrado; el código exacto histórico está archivado para distinguir procedencia de una reproducción posterior.

El margen de parsimonia figura en el candidato fechado antes de la evaluación y su huella original aparece en resultados.json. La relación entre huellas original y portable se registra sin modificar valores experimentales. Es evidencia interna, sin sello de tiempo independiente. [Detalle de reproducción y contrato](../docs/informes/integracion_pr_fase23.md).

In [9]:
fuentes = [
    {"resultado": r.metodo, "fuente": "ejecucion-20261113T184058Z-969d4545 (ZIP de evidencia)",
     "comando manual": "python src/experimentos_seleccion.py", "segundos de etapa registrados": r.segundos}
    for r in resumen_interno.itertuples()
]
fuentes += [
    {"resultado": "Forward max60", "fuente": "forward_max60_artefactos/",
     "comando manual": "python src/forward_tradicional_independiente.py --max-features 60 --salida results/seleccion/reproduccion-forward-max60", "segundos de etapa registrados": progreso_forward["segundos_transcurridos"]},
    {"resultado": "Comparación Validation y baselines", "fuente": "comparacion_validation_congelada/",
     "comando manual": "python src/comparar_selecciones_validation.py --salida results/seleccion/reproduccion-validation", "segundos de etapa registrados": None},
]
print("Tabla 8. Procedencia, regeneración y tiempos registrados")
display(tabla_simple(pd.DataFrame(fuentes)))
print("Duración de comparación: no registrada. Lectura del contrato: cargar_seleccion_definitiva(); no recalcula selección.")

Tabla 8. Procedencia, regeneración y tiempos registrados


,resultado,fuente,comando manual,segundos de etapa registrados
0,spearman,ejecucion-20261113T184058Z-969d4545 (ZIP de evidencia),python src/experimentos_seleccion.py,612.39
1,pearson,ejecucion-20261113T184058Z-969d4545 (ZIP de evidencia),python src/experimentos_seleccion.py,215.67
2,mutua,ejecucion-20261113T184058Z-969d4545 (ZIP de evidencia),python src/experimentos_seleccion.py,216.65
3,lasso,ejecucion-20261113T184058Z-969d4545 (ZIP de evidencia),python src/experimentos_seleccion.py,853.97
4,Forward max60,forward_max60_artefactos/,python src/forward_tradicional_independiente.py --max-features 60 --salida results/seleccion/reproduccion-forward-max60,449.88
5,Comparación Validation y baselines,comparacion_validation_congelada/,python src/comparar_selecciones_validation.py --salida results/seleccion/reproduccion-validation,nan


Duración de comparación: no registrada. Lectura del contrato: cargar_seleccion_definitiva(); no recalcula selección.


## 10. Conclusión

Spearman es oficial por parsimonia dentro del margen previo. Pearson conserva el mínimo absoluto. La próxima fase carga seleccion_definitiva y reutiliza sus reglas; Test permanece reservado. No se certifica la correspondencia de bloques con el calendario físico ni la publicación exacta al cierre. Faltan confirmaciones de Consolidado EE, Consolidado Agua y Totalizadores Energia sobre zona horaria, latencia y revisiones, además de la correspondencia bloque-fecha. Las lecturas con segundos posteriores al minuto de cierre requieren emitir después de recibirlas. La convergencia final de Lasso no elimina una advertencia histórica de un pliegue externo.

[Decisión y limitaciones](../docs/decisions/0015-seleccion-de-variables.md), [guía de conexión](../docs/informes/integracion_pr_fase23.md) y [evidencia conservada](../results/seleccion/historial/README.md).